# MCP Card PDF Builder

Builds a duplex-printable PDF from `<character>_healthy.jpg` / `<character>_injured.jpg`
pairs. Each page holds 2 images (6" x 4" each), stacked vertically and centered on a
US Letter page. Every "front" page (healthy images) is immediately followed by a
"back" page (the matching injured images), so the two land on the same physical
sheet when printed double-sided.

**Alignment assumption:** this layout is a single centered column (not side-by-side),
which means **"Flip on Long Edge"** duplex printing (the common default for portrait
documents) needs no mirroring for front/back to align. If your printer uses
**"Flip on Short Edge"** instead, set `DUPLEX_EDGE = "short"` in the config cell below.

Print just the first sheet double-sided as a test before running the full batch.

## Setup

In [ ]:
from pathlib import Path

from PIL import Image
from reportlab.lib.pagesizes import letter
from reportlab.lib.units import inch
from reportlab.pdfgen import canvas

import re

## Configuration
Adjust these for your project, then run all cells below.

In [ ]:
PROJECT_DIR = Path("/Users/brad/Documents/GitHub/crop_mcp_cards/")
CARDS_DIR = PROJECT_DIR / "output" / "2026_august"
OUTPUT_PDF = CARDS_DIR / "_PDFs" / "august_2026_peerless_only.pdf"
DUPLEX_EDGE = "long"  # "long" (default) or "short" -- see note above

# Optional: path to a text file with one character name per line, no header.
# Only these characters will be included, in the order listed. Set to None
# to include every character found in CARDS_DIR instead.
CHARACTER_LIST_PATH = PROJECT_DIR / "input" / "august_2026_peerless_only.txt"

## Layout constants
6" x 4" images, 2 per page, centered on US Letter.

In [ ]:
PAGE_W, PAGE_H = letter                       # 8.5in x 11in
IMG_W, IMG_H = 6 * inch, 4 * inch
GAP = 0.5 * inch                              # vertical gap between the two images
MARGIN_TB = (PAGE_H - (2 * IMG_H) - GAP) / 2  # top/bottom margin (auto-centered)
X = (PAGE_W - IMG_W) / 2                      # horizontal centering, same for both slots

TOP_Y = PAGE_H - MARGIN_TB - IMG_H
BOTTOM_Y = MARGIN_TB

CROP_MARK_LEN = 0.15 * inch
CROP_MARK_OFFSET = 0.05 * inch

## Find healthy/injured pairs

If `CHARACTER_LIST_PATH` is set, only characters named in that file are included
(in the order listed). Otherwise every character found in `CARDS_DIR` is used.

In [ ]:
def load_character_names(path: Path):
    """Read a single-column, no-header text file of character names,
    applying the same cleaning used when the image filenames were generated."""
    with open(path) as f:
        raw_names = [line.strip() for line in f if line.strip()]

    cleaned_names = []
    for character_name in raw_names:
        name = re.sub(r'[<>:"/\\|?*\x00-\x1f]', '_', character_name)
        name = re.sub(r'[_\s]+', '_', name)
        name = name.strip('_.')
        cleaned_names.append(name)

    return list(dict.fromkeys(cleaned_names))  # de-dupe, preserve file order

def find_pairs(CARDS_DIR: Path, character_names=None):
    """Scan CRDS_DIR for <name>_healthy.jpg / <name>_injured.jpg pairs.

    If character_names is given, only those characters are considered (in the
    order provided), and any name with no matching files at all is reported
    separately from names with only a partial (healthy or injured) match.
    """
    healthy, injured = {}, {}
    files = list(CARDS_DIR.glob("*.jpg")) + list(CARDS_DIR.glob("*.jpeg"))
    for f in files:
        stem = f.stem
        if stem.endswith("_healthy"):
            healthy[stem[: -len("_healthy")]] = f
        elif stem.endswith("_injured"):
            injured[stem[: -len("_injured")]] = f

    if character_names is not None:
        names = character_names
    else:
        names = sorted(set(healthy) | set(injured), key=str.lower)

    complete, incomplete, not_found = [], [], []
    for name in names:
        if name in healthy and name in injured:
            complete.append((name, healthy[name], injured[name]))
        elif name in healthy or name in injured:
            incomplete.append(name)
        else:
            not_found.append(name)

    return complete, incomplete, not_found

In [ ]:


character_names = load_character_names(CHARACTER_LIST_PATH) if CHARACTER_LIST_PATH else None
pairs, incomplete, not_found = find_pairs(CARDS_DIR, character_names)

if not_found:
    print(f"WARNING: {len(not_found)} requested character(s) had no matching files at all:")
    for name in not_found:
        print(f"  - {name}")

if incomplete:
    print(f"WARNING: {len(incomplete)} character(s) missing a healthy/injured pair, skipped:")
    for name in incomplete:
        print(f"  - {name}")

print(f"\n{len(pairs)} complete character pairs found.")

## Drawing helpers

In [ ]:
def draw_crop_marks(c, x, y, w, h):
    """Small corner tick marks just outside an image box, for cutting."""
    o, l = CROP_MARK_OFFSET, CROP_MARK_LEN
    c.setLineWidth(0.5)
    c.setStrokeColorRGB(0.6, 0.6, 0.6)
    corners = [
        (x, y, -1, -1),        # bottom-left
        (x + w, y, 1, -1),     # bottom-right
        (x, y + h, -1, 1),     # top-left
        (x + w, y + h, 1, 1),  # top-right
    ]
    for cx, cy, dx, dy in corners:
        c.line(cx, cy + o * dy, cx, cy + (o + l) * dy)   # vertical tick
        c.line(cx + o * dx, cy, cx + (o + l) * dx, cy)   # horizontal tick


def draw_image_fitted(c, img_path, x, y, w, h):
    """Draw image scaled to fit box w x h, preserving aspect ratio, centered in box."""
    with Image.open(img_path) as im:
        iw, ih = im.size
    scale = min(w / iw, h / ih)
    dw, dh = iw * scale, ih * scale
    dx, dy = x + (w - dw) / 2, y + (h - dh) / 2
    c.drawImage(str(img_path), dx, dy, width=dw, height=dh)
    draw_crop_marks(c, x, y, w, h)

## Build the PDF

In [ ]:
def build_pdf(pairs, output_path, duplex_edge="long"):
    c = canvas.Canvas(str(output_path), pagesize=letter)

    front_slots = [(X, TOP_Y), (X, BOTTOM_Y)]
    back_slots = front_slots if duplex_edge == "long" else list(reversed(front_slots))

    for i in range(0, len(pairs), 2):
        page_pairs = pairs[i:i + 2]  # 1 or 2 characters for this sheet

        # FRONT: healthy images
        for (name, healthy_path, _injured), (sx, sy) in zip(page_pairs, front_slots):
            draw_image_fitted(c, healthy_path, sx, sy, IMG_W, IMG_H)
        c.showPage()

        # BACK: injured images, slotted to align with front once printed duplex
        for (name, _healthy, injured_path), (sx, sy) in zip(page_pairs, back_slots):
            draw_image_fitted(c, injured_path, sx, sy, IMG_W, IMG_H)
        c.showPage()

    c.save()

In [ ]:
if pairs:
    build_pdf(pairs, OUTPUT_PDF, duplex_edge=DUPLEX_EDGE)
    n_sheets = -(-len(pairs) // 2)  # ceil division
    print(f"Done: {len(pairs)} characters -> {n_sheets} sheet(s), "
          f"{n_sheets * 2} PDF pages -> {OUTPUT_PDF}")
else:
    print("No complete healthy/injured pairs found. Nothing to do.")

## Optional: preview the first sheet
Requires `pdf2image` + poppler (you've already got this set up from the card-cropping
project). Renders page 1 (front) and page 2 (back) of the PDF as images so you can
sanity-check the layout before printing.

In [ ]:
try:
    from pdf2image import convert_from_path

    preview_pages = convert_from_path(str(OUTPUT_PDF), dpi=100, first_page=1, last_page=2)
    for i, page_img in enumerate(preview_pages, start=1):
        label = "FRONT" if i == 1 else "BACK"
        print(f"Sheet 1 -- {label}")
        display(page_img)
except ImportError:
    print("pdf2image not installed -- run `pip install pdf2image` (and ensure poppler "
          "is on PATH) to enable this preview.")